In [1]:
from pathlib import Path
import csv
from collections import Counter


bbox_csv_path = Path('../data/raw/RAW_NIH/BBox_List_2017.csv')

with bbox_csv_path.open(newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    abnormality_counts = Counter(row['Finding Label'] for row in reader)

print(f'Total annotated abnormalities: {sum(abnormality_counts.values()):,}')
print('\nAbnormalities per classification:')
for finding_label, count in abnormality_counts.most_common():
    print(f'{finding_label}: {count:,}')


Total annotated abnormalities: 984

Abnormalities per classification:
Atelectasis: 180
Effusion: 153
Cardiomegaly: 146
Infiltrate: 123
Pneumonia: 120
Pneumothorax: 98
Mass: 85
Nodule: 79


In [ ]:
# Keep only these NIH findings, and rename some labels to match your target taxonomy
rename_map = {
    "Effusion": "Pleural effusion",
    "Infiltrate": "Infiltration",
}

keep_labels = {
    "Atelectasis",
    "Cardiomegaly",
    "Pneumothorax",
    "Effusion",
    "Infiltrate",
}

nih_bbox_csv_path = Path("../data/raw/RAW_NIH/BBox_List_2017.csv")
nih_out_csv_path = Path("../data/processed/01_nih_bbox_selected_renamed.csv")
nih_out_csv_path.parent.mkdir(parents=True, exist_ok=True)

total_rows = 0
kept_rows = 0

with nih_bbox_csv_path.open(newline="", encoding="utf-8") as infile, nih_out_csv_path.open(
    "w", newline="", encoding="utf-8"
) as outfile:
    reader = csv.DictReader(infile)

    out_fieldnames = list(reader.fieldnames or [])
    writer = csv.DictWriter(outfile, fieldnames=out_fieldnames)
    writer.writeheader()

    for row in reader:
        total_rows += 1
        label = (row.get("Finding Label") or "").strip()

        # Excludes Mass, Nodule, Pneumonia, etc. by only keeping `keep_labels`
        if label not in keep_labels:
            continue

        row["Finding Label"] = rename_map.get(label, label)  # Effusion->Pleural effusion, Infiltrate->Infiltration
        writer.writerow(row)
        kept_rows += 1

print(f"Read rows: {total_rows:,}")
print(f"Wrote rows (kept+renamed): {kept_rows:,}")
print(f"Output: {nih_out_csv_path.resolve()}")


Read rows: 984
Wrote rows (kept+renamed): 700
Output: D:\Home\Documents\GitHub\cxraide-data-pipeline\notebooks\data\processed\nih_bbox_selected_renamed.csv
